# 03 · Naive RAG
**Concept:** retrieve → stuff into a grounded prompt → generate, with citations.
**TAC scenario:** answering engineer questions from the knowledge base, then finding where Naive RAG breaks.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from rag_common import format_docs, doc_ids, show, golden_set

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a TAC engineer. Answer ONLY from the <context>. Cite sources as [doc_id] after each claim. "
               "If the context does not contain the answer, reply exactly: NOT_IN_KB."),
    ("human", "<context>\n{context}\n</context>\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | model | StrOutputParser()
)

## 1. It works

In [ ]:
q = "Our site-to-site tunnels die for a couple of minutes at exactly the same time every night on 5.1.3. Why?"
print(rag_chain.invoke(q))

## 2. Show the evidence: return sources alongside the answer

In [ ]:
from langchain_core.runnables import RunnableParallel

rag_with_sources = RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(
    answer=lambda x: (prompt | model | StrOutputParser()).invoke({"context": format_docs(x["docs"]), "question": x["question"]})
)
out = rag_with_sources.invoke("A wireless controller keeps rebooting during startup after the 4.8 upgrade. How do we recover?")
print(out["answer"], "\n\nSources:", doc_ids(out["docs"]))

## 3. Where Naive RAG breaks: check retrieval against the golden set

In [ ]:
G = {g["type"]: g for g in reversed(golden_set())}       # one example per question type
for qtype in ["version_specific", "error_code", "multi_hop", "unanswerable"]:
    g = G[qtype]
    docs = retriever.invoke(g["question"])
    print(f"── {qtype}: {g['question']}\n   expected: {g['expected_doc_ids'] or 'nothing (should say NOT_IN_KB)'}")
    show(docs, expected=g["expected_doc_ids"])
    print()

In [ ]:
g = G["unanswerable"]
print(g["question"], "\n→", rag_chain.invoke(g["question"]))

Typical failure points to look for above:
- **wrong version**: a twin article for the other release ranks first
- **error code**: an opaque code like `%STACK-2-SPLIT_DETECTED` is matched weakly by meaning
- **multi-hop**: the KB article is found but the release note with the fix is not
- **noise**: near-duplicates and many similar tickets crowd out other evidence

Notebooks 04-06 fix these one at a time; Notebook 11 measures the difference.

---
**🔧 TAC takeaway:** Naive RAG is the new engineer who reads the first four search hits and answers from those. It's a great baseline, and the failures above show exactly what to fix next.